In [1]:
import pandas as pd
import numpy as np

In [2]:
from hedging import PriceSimulator, GreeksEngine, HedgingBot, delta_hedge_pnl, var_cvar, realized_vol

In [3]:
from data_loader import DataLoader

In [4]:
loader = DataLoader("BTCUSDT", "1h", "2024-01-01", "2026-01-01")

In [5]:
df = loader.fetch()

In [6]:
print(df.head())
print(df.shape)

                        close
time                         
2024-01-01 00:00:00  42475.23
2024-01-01 01:00:00  42613.56
2024-01-01 02:00:00  42581.10
2024-01-01 03:00:00  42330.49
2024-01-01 04:00:00  42399.99
(17545, 1)


In [7]:
vol_series = realized_vol(df, 30*24, 24*365)

In [8]:
vol_series

time
2024-01-01 00:00:00         NaN
2024-01-01 01:00:00         NaN
2024-01-01 02:00:00         NaN
2024-01-01 03:00:00         NaN
2024-01-01 04:00:00         NaN
                         ...   
2025-12-31 20:00:00    0.428218
2025-12-31 21:00:00    0.426634
2025-12-31 22:00:00    0.426609
2025-12-31 23:00:00    0.426463
2026-01-01 00:00:00    0.426413
Name: realized_vol, Length: 17545, dtype: float64

In [9]:
data = pd.concat([df, vol_series], axis=1)

In [10]:
data.dropna(inplace=True)

In [11]:
data.shape

(16825, 2)

In [12]:
data.head()

,close,realized_vol
time,,
2024-01-31 00:00:00,42933.21,0.542396
2024-01-31 01:00:00,42782.82,0.542417
2024-01-31 02:00:00,42883.00,0.542472
2024-01-31 03:00:00,42930.07,0.542092
2024-01-31 04:00:00,42962.64,0.542068


In [20]:
n = len(data)
T = 2
K = 42933
t = np.linspace(0, T, n)
r = 0.03
option_type = "call"
fee_rate = 0.0005

In [21]:
paths = data["close"].values.reshape(1, -1)
sigma_series = data["realized_vol"].values

In [22]:
greek = GreeksEngine(option_type, K, r)

In [23]:
all_deltas = greek.compute_all_deltas(paths, sigma_series, T, t)

In [24]:
all_deltas.shape

(1, 16825)

In [45]:
premium = greek.price(paths[0, 0], sigma_series[0], T, t[0])

In [46]:
premium

np.float64(13736.102831961145)

In [47]:
delta_hedge_pnl(paths, all_deltas, premium, r, T, t, fee_rate, K, option_type)

array([504.30275042])